In [1]:
# 基于LangChain的大语言模型应用--基于文档问答
# 目的：使用LLM回答关于提供的文档的问题
# 从本章节开始会开始设计Embedding模型和向量存储(Vector Stores)
# embedding模型会把一段文字翻译成一段向量，意思越相近，向量值越接近
# 注意deepseek本身不是embedding模型，因此在此处使用的是阿里云的模型
# ollama pull ryanshillington/Qwen3-Embedding-8B
import os
api_key = os.environ.get("DEEPSEEK_API_KEY")
qwen_api_key = os.environ.get("QWEN_API_KEY")

In [38]:
# 导入构建链必备的库
from langchain_classic.chains import RetrievalQA # 帮助检索文档
from langchain_openai.chat_models import ChatOpenAI
from langchain_classic.document_loaders import CSVLoader # 文件加载器
from langchain_classic.vectorstores import DocArrayInMemorySearch # 向量存储，使用内存进行存储
from IPython.display import display, Markdown

In [3]:
llm = ChatOpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com/",
    model="deepseek-v4-flash"
)

In [4]:
file = "OutdoorClothingCatalog_1000.csv"
loader = CSVLoader(file_path=file,encoding="utf-8")

In [7]:
# 导入索引，帮助创建向量存储
from langchain_classic.indexes import  VectorstoreIndexCreator
from langchain_openai import OpenAIEmbeddings

In [12]:
embedding = OpenAIEmbeddings(
    api_key=qwen_api_key,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen3.7-text-embedding",
    check_embedding_ctx_length=False, # 发送原始文本
    # DashScope单词最多运行20条
    chunk_size=20
)

index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch,
    embedding=embedding
).from_loaders([loader]) # 调用文档加载器，传入包含加载器的列表

In [13]:
query = "Please list all your shirts with sun protection \
in a table in markdown and summarize each one."

In [14]:
response = index.query(query, llm=llm)

In [15]:
response

"| Name | Summary |\n|------|---------|\n| Men's Plaid Tropic Shirt, Short-Sleeve | Ultracomfortable, wrinkle-free, quick-drying shirt with UPF 50+ protection (blocks 98% UV rays). Made from 52% polyester/48% nylon, machine washable, with front/back cape venting and two bellows pockets. Designed for hot weather and travel. |\n| Sun Shield Shirt | High-performance, slightly fitted sun shirt with UPF 50+ (SPF 50+) protection. Made from 78% nylon/22% Lycra Xtra Life fiber for abrasion resistance and moisture-wicking comfort. Handwash and line dry; fits over swimsuits; recommended by The Skin Cancer Foundation. |\n| Men's TropicVibe Shirt, Short-Sleeve | Traditional fit, lightweight sun-protection shirt with UPF 50+, made from 71% nylon/29% polyester with a polyester mesh lining. Wrinkle resistant, features cape venting and two front bellows pockets; machine washable and dryable. |\n| Men's Tropical Plaid Short-Sleeve Shirt | Lightest hot-weather shirt with UPF 50+ protection. 100% polyest

In [39]:
display(Markdown(response))

| Name | Summary |
|------|---------|
| Men's Plaid Tropic Shirt, Short-Sleeve | Ultracomfortable, wrinkle-free, quick-drying shirt with UPF 50+ protection (blocks 98% UV rays). Made from 52% polyester/48% nylon, machine washable, with front/back cape venting and two bellows pockets. Designed for hot weather and travel. |
| Sun Shield Shirt | High-performance, slightly fitted sun shirt with UPF 50+ (SPF 50+) protection. Made from 78% nylon/22% Lycra Xtra Life fiber for abrasion resistance and moisture-wicking comfort. Handwash and line dry; fits over swimsuits; recommended by The Skin Cancer Foundation. |
| Men's TropicVibe Shirt, Short-Sleeve | Traditional fit, lightweight sun-protection shirt with UPF 50+, made from 71% nylon/29% polyester with a polyester mesh lining. Wrinkle resistant, features cape venting and two front bellows pockets; machine washable and dryable. |
| Men's Tropical Plaid Short-Sleeve Shirt | Lightest hot-weather shirt with UPF 50+ protection. 100% polyester, wrinkle-resistant, traditional fit, with front/back cape venting and two front bellows pockets. Machine washable; highest rated sun protection possible. |